In [0]:
%pip install meteostat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 86.9 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2024.1
    Not uninstalling pytz at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b1d6dc01-853a-4629-b189-e5fdf6bf6c44
    Can't uninstall 'pytz'. No files were found to uninstall.
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Not uninstalling pandas at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-b1d6dc01-853a-4629-b189-e5fdf6bf6c44
    Can't uninstall 'pandas'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-connect 18.0.2 requires pandas<3,>=1.0.5, but you have pandas 3.0.2 which is incompatible.
Note: y

In [0]:
dbutils.library.restartPython()

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/flight_data/mllib_tmp"

from datetime import datetime
import pandas as pd
import meteostat as ms
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)

STATIONS = {
    "JFK": "74486",
    "LGA": "72503",
    "EWR": "72502",
    "ALB": "72518",
    "BUF": "72528",
}

START = datetime(2024, 12, 1)
END   = datetime(2025, 11, 30, 23, 59)

BRONZE_WEATHER = "bronze_weather"
SILVER_WEATHER = "silver_weather"
GOLD_WEATHER   = "gold_weather_features"

print(f"Meteostat version: {ms.__version__ if hasattr(ms, '__version__') else 'unknown'}")
print("Setup done.")

Meteostat version: 2.1.4
Setup done.


In [0]:
bronze_schema = StructType([
    StructField("station_code", StringType(),    False),
    StructField("station_id",   StringType(),    False),
    StructField("obs_time",     TimestampType(), False),
    StructField("temp",         DoubleType(),    True),
    StructField("dwpt",         DoubleType(),    True),
    StructField("rhum",         DoubleType(),    True),
    StructField("prcp",         DoubleType(),    True),
    StructField("snow",         DoubleType(),    True),
    StructField("wdir",         DoubleType(),    True),
    StructField("wspd",         DoubleType(),    True),
    StructField("wpgt",         DoubleType(),    True),
    StructField("pres",         DoubleType(),    True),
    StructField("coco",         DoubleType(),    True),
])

spark.sql(f"DROP TABLE IF EXISTS {BRONZE_WEATHER}")

all_cols = ["temp", "dwpt", "rhum", "prcp", "snow", "wdir", "wspd", "wpgt", "pres", "coco"]

def fetch_station(station_id, start, end):
    try:
        return ms.hourly(ms.Station(id=station_id), start, end).fetch()
    except AttributeError:
        from meteostat import Hourly
        return Hourly(station_id, start, end).fetch()

for station_code, station_id in STATIONS.items():
    print(f"Fetching {station_code} ({station_id})...")
    pdf = fetch_station(station_id, START, END)
    pdf = pdf.reset_index()
    time_col = "time" if "time" in pdf.columns else pdf.columns[0]
    pdf = pdf.rename(columns={time_col: "obs_time"})
    pdf["station_code"] = station_code
    pdf["station_id"]   = station_id
    for c in all_cols:
        if c not in pdf.columns:
            pdf[c] = None
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
    pdf = pdf[["station_code", "station_id", "obs_time"] + all_cols]
    sdf = spark.createDataFrame(pdf, schema=bronze_schema)
    (
        sdf.write
           .format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(BRONZE_WEATHER)
    )
    print(f"  {station_code}: {len(pdf):,} hourly rows written")

print("\nBronze complete.")

Fetching JFK (74486)...
  JFK: 8,760 hourly rows written
Fetching LGA (72503)...
  LGA: 8,760 hourly rows written
Fetching EWR (72502)...
  EWR: 8,760 hourly rows written
Fetching ALB (72518)...
  ALB: 8,760 hourly rows written
Fetching BUF (72528)...
  BUF: 8,760 hourly rows written

Bronze complete.


In [0]:
spark.sql(f"""
    SELECT station_code,
           COUNT(*)             AS n_rows,
           MIN(obs_time)        AS earliest,
           MAX(obs_time)        AS latest
    FROM {BRONZE_WEATHER}
    GROUP BY station_code
    ORDER BY station_code
""").show(truncate=False)

+------------+------+-------------------+-------------------+
|station_code|n_rows|earliest           |latest             |
+------------+------+-------------------+-------------------+
|ALB         |8760  |2024-12-01 00:00:00|2025-11-30 23:00:00|
|BUF         |8760  |2024-12-01 00:00:00|2025-11-30 23:00:00|
|EWR         |8760  |2024-12-01 00:00:00|2025-11-30 23:00:00|
|JFK         |8760  |2024-12-01 00:00:00|2025-11-30 23:00:00|
|LGA         |8760  |2024-12-01 00:00:00|2025-11-30 23:00:00|
+------------+------+-------------------+-------------------+



In [0]:
bronze = spark.table(BRONZE_WEATHER)

cleaned = (
    bronze
        .drop("wpgt", "coco", "snow")
        .withColumn("prcp", F.coalesce(F.col("prcp"), F.lit(0.0)))
        .withColumn("obs_hour_ts", F.date_trunc("hour", F.col("obs_time")))
)

airport_map_rows = [
    ("JFK", "JFK"),
    ("LGA", "LGA"),
    ("HPN", "LGA"),
    ("ISP", "LGA"),
    ("EWR", "EWR"),
    ("ALB", "ALB"),
    ("BGM", "ALB"),
    ("ELM", "ALB"),
    ("ITH", "ALB"),
    ("SWF", "ALB"),
    ("BUF", "BUF"),
    ("ROC", "BUF"),
    ("SYR", "BUF"),
    ("IAG", "BUF"),
]
airport_map = spark.createDataFrame(airport_map_rows, ["airport", "station_code"])

expanded = (
    cleaned.join(airport_map, on="station_code", how="inner")
           .select(
               "airport",
               "station_code",
               "station_id",
               "obs_hour_ts",
               "temp", "dwpt", "rhum",
               "prcp",
               "wdir", "wspd",
               "pres"
           )
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_WEATHER}")
(
    expanded.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(SILVER_WEATHER)
)

print(f"Silver rows: {spark.table(SILVER_WEATHER).count():,}")
spark.table(SILVER_WEATHER).printSchema()

Silver rows: 122,640
root
 |-- airport: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- obs_hour_ts: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- dwpt: double (nullable = true)
 |-- rhum: double (nullable = true)
 |-- prcp: double (nullable = true)
 |-- wdir: double (nullable = true)
 |-- wspd: double (nullable = true)
 |-- pres: double (nullable = true)



In [0]:
spark.sql(f"""
    SELECT airport, COUNT(*) AS n_hours
    FROM {SILVER_WEATHER}
    GROUP BY airport
    ORDER BY airport
""").show(20, truncate=False)

+-------+-------+
|airport|n_hours|
+-------+-------+
|ALB    |8760   |
|BGM    |8760   |
|BUF    |8760   |
|ELM    |8760   |
|EWR    |8760   |
|HPN    |8760   |
|IAG    |8760   |
|ISP    |8760   |
|ITH    |8760   |
|JFK    |8760   |
|LGA    |8760   |
|ROC    |8760   |
|SWF    |8760   |
|SYR    |8760   |
+-------+-------+



In [0]:
silver = spark.table(SILVER_WEATHER)

gold = (
    silver
        .withColumn("is_precipitating", (F.col("prcp") > 0).cast("int"))
        .withColumn("heavy_precip",     (F.col("prcp") > 2.5).cast("int"))
        .withColumn("freezing",         (F.col("temp") < 0.0).cast("int"))
        .withColumn("high_wind",        (F.col("wspd") > 30.0).cast("int"))
        .withColumn("very_high_wind",   (F.col("wspd") > 50.0).cast("int"))
)

gold = gold.withColumn(
    "severe_weather",
    (
        (F.col("heavy_precip") == 1) |
        (F.col("very_high_wind") == 1) |
        ((F.col("freezing") == 1) & (F.col("is_precipitating") == 1))
    ).cast("int")
)

spark.sql(f"DROP TABLE IF EXISTS {GOLD_WEATHER}")
(
    gold.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_WEATHER)
)

print(f"Gold rows: {spark.table(GOLD_WEATHER).count():,}")
spark.table(GOLD_WEATHER).printSchema()

Gold rows: 122,640
root
 |-- airport: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- obs_hour_ts: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- dwpt: double (nullable = true)
 |-- rhum: double (nullable = true)
 |-- prcp: double (nullable = true)
 |-- wdir: double (nullable = true)
 |-- wspd: double (nullable = true)
 |-- pres: double (nullable = true)
 |-- is_precipitating: integer (nullable = true)
 |-- heavy_precip: integer (nullable = true)
 |-- freezing: integer (nullable = true)
 |-- high_wind: integer (nullable = true)
 |-- very_high_wind: integer (nullable = true)
 |-- severe_weather: integer (nullable = true)



In [0]:
spark.sql(f"""
    SELECT
        ROUND(100.0 * SUM(is_precipitating) / COUNT(*), 2) AS pct_precip_hours,
        ROUND(100.0 * SUM(heavy_precip)     / COUNT(*), 2) AS pct_heavy_precip,
        ROUND(100.0 * SUM(freezing)         / COUNT(*), 2) AS pct_freezing,
        ROUND(100.0 * SUM(high_wind)        / COUNT(*), 2) AS pct_high_wind,
        ROUND(100.0 * SUM(very_high_wind)   / COUNT(*), 2) AS pct_very_high_wind,
        ROUND(100.0 * SUM(severe_weather)   / COUNT(*), 2) AS pct_severe_weather
    FROM {GOLD_WEATHER}
""").show(truncate=False)

+----------------+----------------+------------+-------------+------------------+------------------+
|pct_precip_hours|pct_heavy_precip|pct_freezing|pct_high_wind|pct_very_high_wind|pct_severe_weather|
+----------------+----------------+------------+-------------+------------------+------------------+
|7.55            |0.74            |16.07       |7.33         |0.09              |1.65              |
+----------------+----------------+------------+-------------+------------------+------------------+

